In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
rahman4li_business_process_analysis_and_modeling_car_repair_path = kagglehub.dataset_download('rahman4li/business-process-analysis-and-modeling-car-repair')

print('Data source import complete.')


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import os
from IPython.display import display

def get_dataset_root():
    """Finds the dataset root directory dynamically."""
    base_dir = '/kaggle/input'
    for root, dirs, files in os.walk(base_dir):
        if any(f.endswith(('.bpmn', '.pnml')) for f in files):
            return root
    return None

def detailed_analyze_bpmn(file_path):
    """Detailed extraction of participants and process activities from BPMN."""
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        ns = {'bpmn': 'http://www.omg.org/spec/BPMN/20100524/MODEL'}

        participants = [p.get('name') for p in root.findall('.//bpmn:participant', ns) if p.get('name')]

        activities = []
        task_types = ['task', 'userTask', 'serviceTask', 'manualTask']
        for t_type in task_types:
            for task in root.findall(f'.//bpmn:{t_type}', ns):
                name = task.get('name')
                if name:
                    activities.append({'Component': t_type.replace('Task', ' Task'), 'Activity Name': name})

        return participants, pd.DataFrame(activities)
    except:
        return [], pd.DataFrame()

def detailed_analyze_pnml(file_path):
    """Detailed analysis of Petri Net structure (States and Transitions)."""
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        places = []
        for p in root.findall('.//place'):
            name_node = p.find('./name/text')
            places.append(name_node.text if name_node is not None else p.get('id'))

        transitions = []
        for t in root.findall('.//transition'):
            name_node = t.find('./name/text')
            if name_node is not None and name_node.text:
                transitions.append(name_node.text)

        return pd.DataFrame(places, columns=['States (Places)']), pd.DataFrame(transitions, columns=['Actions (Transitions)'])
    except:
        return pd.DataFrame(), pd.DataFrame()

# --- Main Analysis Logic ---
root_path = get_dataset_root()

if root_path:
    files = sorted(os.listdir(root_path))

    for f in files:
        full_p = os.path.join(root_path, f)

        if f.endswith('.bpmn') or f.endswith('.pnml'):
            print("-" * 60)
            print(f"ANALYSIS REPORT: {f.upper()}")
            print("-" * 60)

            if f.endswith('.bpmn'):
                parts, act_df = detailed_analyze_bpmn(full_p)
                print(f"Entities: {', '.join(parts) if parts else 'Main Process'}")
                if not act_df.empty:
                    print("\nProcess Activities:")
                    display(act_df)
                else:
                    print("\nNo specific activities found in this BPMN model.")

            elif f.endswith('.pnml'):
                places_df, trans_df = detailed_analyze_pnml(full_p)
                print(f"Structural Summary: {len(places_df)} States, {len(trans_df)} Transitions")

                if not trans_df.empty:
                    print("\nModel Transitions:")
                    display(trans_df)

                # Contextual info for redesign/complete models
                if 'redesign' in f.lower():
                    print("\nNote: This model represents the optimized (redesigned) workflow.")
                elif 'complete' in f.lower():
                    print("\nNote: This model represents the end-to-end integrated process.")

            print("\n")
else:
    print("Error: Dataset directory not found. Please verify data attachment.")